# DeepAlgPro — encoder ablation: ProtT5 vs one-hot

| row | encoder | architecture |
|-----|---------|--------------|
| 1 | one-hot (21) | convATTnet, unchanged |
| 2 | ProtT5-XL (1024) | convATTnet, unchanged |

Identical hyperparameters, first 5 of 10 folds, same test set (n=1420).
Published baseline: `0.91620 accuracy.`


In [1]:
%cd /content
!rm -rf deep-alg-pro
!git clone -q --depth 1 -b lab https://github.com/keizandrea/deep-alg-pro.git
%cd /content/deep-alg-pro
!pip -q install sentencepiece transformers
!git log --oneline -1


/content
/content/deep-alg-pro
edc0f57 (grafted, HEAD -> lab, origin/lab, origin/HEAD) Add --tag and --folds; save checkpoints to weights/


In [2]:
!grep -c "EmbDataset\|read_ids" Train.py
!grep -n "k >= args.folds\|tag = args.tag" Train.py
!grep -n "'--tag'\|'--folds'\|'--in-channels'" main.py
!python -c "import torch; print('GPU:', torch.cuda.get_device_name(0))"


4
131:    tag = args.tag or args.model
198:        if k >= args.folds:
24:    argparser.add_argument('--in-channels', default=21, type=int,
28:    argparser.add_argument('--tag', default=None, type=str,
30:    argparser.add_argument('--folds', default=10, type=int,
GPU: Tesla T4


In [4]:
!pip install Bio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 68.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [Bio]


In [5]:
from Bio import SeqIO
SeqIO.write(list(SeqIO.parse('data/all.train.fasta','fasta'))[:20], 'mini.fasta', 'fasta')


20

In [6]:
!python embed.py --fasta mini.fasta --out emb/smoke --plm Rostlab/prot_t5_xl_half_uniref50-enc --batch 4


tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 84.4kB/s]
spiece.model: 100% 238k/238k [00:00<00:00, 17.6MB/s]
special_tokens_map.json: 100% 1.79k/1.79k [00:00<00:00, 5.48MB/s]
config.json: 100% 656/656 [00:00<00:00, 3.28MB/s]

pytorch_model.bin: downloading bytes:  11% 271M/2.42G [00:01<00:08, 243MB/s, 24.3MB/s  ]
pytorch_model.bin: downloading bytes:  13% 312M/2.42G [00:01<00:09, 228MB/s, 26.5MB/s  ]
pytorch_model.bin: downloading bytes:  20% 472M/2.42G [00:02<00:05, 325MB/s, 39.1MB/s  ]
pytorch_model.bin: downloading bytes:  26% 618M/2.42G [00:02<00:05, 336MB/s, 50.8MB/s  ]
pytorch_model.bin: downloading bytes:  33% 788M/2.42G [00:03<00:04, 368MB/s, 66.5MB/s  ]
pytorch_model.bin: downloading bytes:  60% 1.46G/2.42G [00:04<00:02, 393MB/s,  114MB/s  ]
pytorch_model.bin: downloading bytes:  74% 1.80G/2.42G [00:08<00:05, 110MB/s,  114MB/s  ]
pytorch_model.bin: downloading bytes:  82% 1.99G/2.42G [00:11<00:03, 113MB/s, 97.2MB/s  ] 
pytorch_model.bin: downloading bytes:  90% 2.18G/2.4

In [8]:
!python embed.py --fasta data/all.train.fasta --out emb/prott5/train --plm Rostlab/prot_t5_xl_half_uniref50-enc --batch 4
!python embed.py --fasta data/all.test.fasta  --out emb/prott5/test  --plm Rostlab/prot_t5_xl_half_uniref50-enc --batch 4


Loading weights: 100% 196/196 [00:01<00:00, 123.37it/s]
5680 seqs, length 11-999
4/5680
84/5680
164/5680
244/5680
324/5680
404/5680
484/5680
564/5680
644/5680
724/5680
804/5680
884/5680
964/5680
1044/5680
1124/5680
1204/5680
1284/5680
1364/5680
1444/5680
1524/5680
1604/5680
1684/5680
1764/5680
1844/5680
1924/5680
2004/5680
2084/5680
2164/5680
2244/5680
2324/5680
2404/5680
2484/5680
2564/5680
2644/5680
2724/5680
2804/5680
2884/5680
2964/5680
3044/5680
3124/5680
3204/5680
3284/5680
3364/5680
3444/5680
3524/5680
3604/5680
3684/5680
3764/5680
3844/5680
3924/5680
4004/5680
4084/5680
4164/5680
4244/5680
4324/5680
4404/5680
4484/5680
4564/5680
4644/5680
4724/5680
4804/5680
4884/5680
4964/5680
5044/5680
5124/5680
5204/5680
5284/5680
5364/5680
5444/5680
[W1111 18:19:29.528610756 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 165675008 bytes (free: 14483456, total: 15637086208).
5524/5680
5604/5680
[W1111 18:20:24.207076373 CUDACachingAlloca

In [9]:
import os, glob
from Bio import SeqIO
for split in ['train', 'test']:
    ids  = [r.id for r in SeqIO.parse(f'data/all.{split}.fasta', 'fasta')]
    have = {os.path.basename(f)[:-4] for f in glob.glob(f'emb/prott5/{split}/*.npy')}
    print(f'{split:5s}  {len(have):5d} / {len(ids):5d}   missing {len([i for i in ids if i not in have])}')


train   5680 /  5680   missing 0
test    1420 /  1420   missing 0


In [10]:
!mv test.log results/logs_test_baseline.log


In [14]:
!pip -q install -r requirements.txt sentencepiece
import torch; print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())

torch: 2.14.1+cu130 | cuda: True


In [15]:
!python main.py -i data/all.train.fasta --mode train --tag onehot_base --folds 5 -b 32 --epochs 30

Output streaming akan dipotong hingga 5000 baris terakhir.
[k: 4, batch: 125] train_loss: 0.248
[k: 4, batch: 126] train_loss: 0.370
[k: 4, batch: 127] train_loss: 0.428
[k: 4, batch: 128] train_loss: 0.304
[k: 4, batch: 129] train_loss: 0.237
[k: 4, batch: 130] train_loss: 0.257
[k: 4, batch: 131] train_loss: 0.252
[k: 4, batch: 132] train_loss: 0.253
[k: 4, batch: 133] train_loss: 0.200
[k: 4, batch: 134] train_loss: 0.375
[k: 4, batch: 135] train_loss: 0.260
[k: 4, batch: 136] train_loss: 0.321
[k: 4, batch: 137] train_loss: 0.332
[k: 4, batch: 138] train_loss: 0.186
[k: 4, batch: 139] train_loss: 0.154
[k: 4, batch: 140] train_loss: 0.287
[k: 4, batch: 141] train_loss: 0.266
[k: 4, batch: 142] train_loss: 0.263
[k: 4, batch: 143] train_loss: 0.344
[k: 4, batch: 144] train_loss: 0.413
[k: 4, batch: 145] train_loss: 0.087
[k: 4, batch: 146] train_loss: 0.314
[k: 4, batch: 147] train_loss: 0.352
[k: 4, batch: 148] train_loss: 0.195
[k: 4, batch: 149] train_loss: 0.142
[k: 4, batch: 15

In [16]:
!for k in 1 2 3 4 5; do python main.py -i data/all.test.fasta --mode test -b 64 --weights weights/onehot_base.$k.pt; done
!cat test.log


There are 1 GPU(s) available.
We will use the GPU: Tesla T4
There are 1 GPU(s) available.
We will use the GPU: Tesla T4
convATTnet (weights/onehot_base.1.pt)  F1: 0.87707	Accuracy: 0.87465	Precision: 0.86043	Recall: 0.89437	AUROC: 0.94330
There are 1 GPU(s) available.
We will use the GPU: Tesla T4
There are 1 GPU(s) available.
We will use the GPU: Tesla T4
convATTnet (weights/onehot_base.2.pt)  F1: 0.86922	Accuracy: 0.86268	Precision: 0.82971	Recall: 0.91268	AUROC: 0.94077
There are 1 GPU(s) available.
We will use the GPU: Tesla T4
There are 1 GPU(s) available.
We will use the GPU: Tesla T4
convATTnet (weights/onehot_base.3.pt)  F1: 0.88110	Accuracy: 0.87817	Precision: 0.86040	Recall: 0.90282	AUROC: 0.95255
There are 1 GPU(s) available.
We will use the GPU: Tesla T4
There are 1 GPU(s) available.
We will use the GPU: Tesla T4
convATTnet (weights/onehot_base.4.pt)  F1: 0.86765	Accuracy: 0.86056	Precision: 0.82570	Recall: 0.91408	AUROC: 0.94184
There are 1 GPU(s) available.
We will use th

In [17]:
import numpy as np
for k in [1,2,3,4,5]:
    d   = np.loadtxt(f'onehot_base.{k}.fold.everyepoch.valid.txt')
    acc, f1 = d[:,2], d[:,6]
    print(f'fold {k}:  best acc {acc.max():.4f} @ ep {acc.argmax():2d}   '
          f'best f1 @ ep {f1.argmax():2d}   ep0-4 {acc[:5].mean():.4f} -> ep25-29 {acc[-5:].mean():.4f}')


fold 1:  best acc 0.8511 @ ep 29   best f1 @ ep 29   ep0-4 0.7096 -> ep25-29 0.8471
fold 2:  best acc 0.8695 @ ep 27   best f1 @ ep 29   ep0-4 0.7346 -> ep25-29 0.8596
fold 3:  best acc 0.8897 @ ep 29   best f1 @ ep 27   ep0-4 0.7651 -> ep25-29 0.8857
fold 4:  best acc 0.8860 @ ep 27   best f1 @ ep 27   ep0-4 0.7430 -> ep25-29 0.8691
fold 5:  best acc 0.8805 @ ep 29   best f1 @ ep 29   ep0-4 0.7143 -> ep25-29 0.8636


## Train

In [18]:
%cd /content/deep-alg-pro
!git fetch --depth 1 origin lab && git reset --hard origin/lab
!grep -n patience main.py Train.py
!ls emb/prott5/train | wc -l && ls emb/prott5/test | wc -l

/content/deep-alg-pro
remote: Total 0 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
From https://github.com/keizandrea/deep-alg-pro
 * branch            lab        -> FETCH_HEAD
HEAD is now at edc0f57 Add --tag and --folds; save checkpoints to weights/
5680
1420


In [20]:
!git fetch --depth 1 origin lab && git reset --hard origin/lab
!python main.py --help | grep -A1 patience


remote: Enumerating objects: 10, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 6 (delta 4), reused 6 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 893 bytes | 111.00 KiB/s, done.
From https://github.com/keizandrea/deep-alg-pro
 * branch            lab        -> FETCH_HEAD
 + edc0f57...4f7e099 lab        -> origin/lab  (forced update)
HEAD is now at 4f7e099 Add --patience early stopping; record results so far
               [--patience PATIENCE]

--
  --patience PATIENCE   stop a fold after N epochs with no F1 gain; 0 = never


In [21]:
!python main.py -i data/all.train.fasta --mode train --tag onehot_e100 --folds 5 -b 32 --epochs 100 --patience 15


Output streaming akan dipotong hingga 5000 baris terakhir.
[k: 5, batch: 125] train_loss: 0.246
[k: 5, batch: 126] train_loss: 0.233
[k: 5, batch: 127] train_loss: 0.287
[k: 5, batch: 128] train_loss: 0.304
[k: 5, batch: 129] train_loss: 0.268
[k: 5, batch: 130] train_loss: 0.133
[k: 5, batch: 131] train_loss: 0.266
[k: 5, batch: 132] train_loss: 0.158
[k: 5, batch: 133] train_loss: 0.330
[k: 5, batch: 134] train_loss: 0.154
[k: 5, batch: 135] train_loss: 0.292
[k: 5, batch: 136] train_loss: 0.141
[k: 5, batch: 137] train_loss: 0.240
[k: 5, batch: 138] train_loss: 0.351
[k: 5, batch: 139] train_loss: 0.294
[k: 5, batch: 140] train_loss: 0.224
[k: 5, batch: 141] train_loss: 0.291
[k: 5, batch: 142] train_loss: 0.232
[k: 5, batch: 143] train_loss: 0.200
[k: 5, batch: 144] train_loss: 0.280
[k: 5, batch: 145] train_loss: 0.269
[k: 5, batch: 146] train_loss: 0.399
[k: 5, batch: 147] train_loss: 0.237
[k: 5, batch: 148] train_loss: 0.243
[k: 5, batch: 149] train_loss: 0.555
[k: 5, batch: 15

In [22]:
!for k in 1 2 3 4 5; do python main.py -i data/all.test.fasta --mode test -b 64 --weights weights/onehot_e100.$k.pt; done
!tail -5 test.log


There are 1 GPU(s) available.
We will use the GPU: Tesla T4
There are 1 GPU(s) available.
We will use the GPU: Tesla T4
convATTnet (weights/onehot_e100.1.pt)  F1: 0.90398	Accuracy: 0.90141	Precision: 0.88102	Recall: 0.92817	AUROC: 0.96353
There are 1 GPU(s) available.
We will use the GPU: Tesla T4
There are 1 GPU(s) available.
We will use the GPU: Tesla T4
convATTnet (weights/onehot_e100.2.pt)  F1: 0.89834	Accuracy: 0.90070	Precision: 0.92024	Recall: 0.87746	AUROC: 0.95641
There are 1 GPU(s) available.
We will use the GPU: Tesla T4
There are 1 GPU(s) available.
We will use the GPU: Tesla T4
convATTnet (weights/onehot_e100.3.pt)  F1: 0.90922	Accuracy: 0.90775	Precision: 0.89495	Recall: 0.92394	AUROC: 0.96439
There are 1 GPU(s) available.
We will use the GPU: Tesla T4
There are 1 GPU(s) available.
We will use the GPU: Tesla T4
convATTnet (weights/onehot_e100.4.pt)  F1: 0.90442	Accuracy: 0.90563	Precision: 0.91618	Recall: 0.89296	AUROC: 0.96251
There are 1 GPU(s) available.
We will use th

In [23]:
!for k in 1 2 3 4 5; do echo -n "fold $k best epoch: "; sort -k7 -g onehot_e100.$k.fold.everyepoch.valid.txt | tail -1 | cut -f1; done


fold 1 best epoch: 68
fold 2 best epoch: 42
fold 3 best epoch: 76
fold 4 best epoch: 59
fold 5 best epoch: 50
